> **Redacted copy for aaronsuu.github.io.** *Dataset not included: it was provided for course use only and contains sensitive responses. Notebook outputs that showed raw data have been removed.* Code and charts are unchanged; the notebook will not run without the data.

# Introduction:
This Notebook uses BERTopic to explore the TA Support Data (Homework 5) for potential topics that characterize those responses. Minimal preprocessing is applied in this notebook (convert text to lower case, expand contractions, and remove punctuation). Each document is one student response.

# Install Non-Standard Libraries
The following cell only needs to be run once per session.

In [19]:
!pip install bertopic
!pip install contractions
!pip install -U sentence-transformers

# Load Data

In [20]:
#Load the Data into a dataframe
#Before loading data, spell check the data to correct spelling errors
#and perform other necessary cleaning tasks

import os
import pandas as pd

# Load dataset
file_path = 'content/TASupport_Data.xlsx' 
data = pd.read_excel(file_path)



In [21]:
#TA Support Data has no Assigned Section column, so all documents are kept
data_filtered = data.copy()

In [22]:
#Covert Text to Lower Case

#This step in data preparation and preprocessing converts all text to lower case
#To avoid any problems with case-senstitive processing and topic modelling in subsequent cells

#Convert all text to lower case
data_filtered.loc[:, 'LowerCase'] = data_filtered['Document'].str.lower()

#Print  Lower Case Data for Sanity Checking
print (data_filtered['LowerCase'])

In [23]:
import contractions

#Expand Contractions
#This step expands Contractions (e.g. "don't" to "do not")

#Define function that expands contractions in a single string of text
def expand_contractions_text(text):
    if isinstance(text, str):
        # contractions.fix() handles the entire string/sentence
        expanded_text = contractions.fix(text)
        return expanded_text
    return text

#Apply the function to all of the texts in "LowerCase" column of the dataframe
data_filtered['ExpandContractions'] = data_filtered['LowerCase'].apply(expand_contractions_text)

#Print Expaneded Data for Sanity Checking
print (data_filtered['ExpandContractions'])

In [24]:
#Remove Punctuation
import string

#Define function that removes punctuation in a single string of text
def remove_punctuation(text):
    """Removes punctuation from a string."""
    return ''.join([char for char in text if char not in string.punctuation])

#Apply the function to all of the texts in "ExpandContractions" column of the dataframe
data_filtered['PreProcessed Sentence'] = data_filtered['ExpandContractions'].apply(remove_punctuation)

#Print Data with Punctuation Removed for Sanity Checking
print (data_filtered['PreProcessed Sentence'])

# BERTopic Modelling
This series of cells allows for modelling the abstract sentences using four different options:  (1) a BERTopic model with all parameters set to their default values; (2) a BERTopic model that forces BERTopic to create more clusters or smaller clusters by setting the min_cluster_size parameter to a small number; (3) Option (2) plus adjusting min_samples to a smaller number than min_cluster_size in order to reduce the number of documents assigned as outliers; and (4) an option to seed the topic list with apriori information about the topics contained in the data. Seed topics are a nudge not a rule, meaning that if a seed topic does not actually appear in your dataset—or if your data naturally splits that theme into smaller sub-topics, then BERTopic will ignore or split the seed rather than forcing a false cluster.

In [25]:
#Generate BERTopic Model for Exploring/Modelling Article Abstracts
#Select from four options (uncomment one option and comment out the rest)

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

#Assign docs to the values of PreProcessed Sentences from the input dataset
docs = data_filtered['PreProcessed Sentence']
#Sanity check to make sure the correct docs dataframe has been created
print (docs)

##Option 1:  Straightforward BERTopic Modelling with no parameter adjustments
# topic_model = BERTopic(verbose=True)
# topics, probabilities = topic_model.fit_transform(docs)

##Option 2:  Force BERTopic to create more clusters or smaller clusters by adjusting min_cluster_size
##Sets min_cluster_size to force smaller clusters
##Note that by default min_samples is set equal to min_cluster_size
hdbscan_model = HDBSCAN(
    min_cluster_size=10,
    prediction_data=True,
    )
vectorizer_model = CountVectorizer(stop_words="english")
topic_model = BERTopic(hdbscan_model=hdbscan_model, vectorizer_model=vectorizer_model, embedding_model="all-MiniLM-L6-v2")
topics, probs = topic_model.fit_transform(docs)

#Option 3
#Lowering the value of min_samples allows looser clusters to form, thus reducing outliers
# hdbscan_model = HDBSCAN(
#     min_cluster_size=5,
#     min_samples = 1,
#     prediction_data=True,
#     )
# vectorizer_model = CountVectorizer(stop_words="english")
# topic_model = BERTopic(hdbscan_model=hdbscan_model, vectorizer_model=vectorizer_model, embedding_model="all-MiniLM-L6-v2")
# topics, probs = topic_model.fit_transform(docs)

##Option 4:
#seed_topic_list = [["belonging", "identity"],["artificial intelligence"],["self efficacy"]]
#vectorizer_model = CountVectorizer(stop_words="english")
#topic_model = BERTopic(hdbscan_model=hdbscan_model, vectorizer_model=vectorizer_model, embedding_model="all-MiniLM-L6-v2")
##Fit the model and transform the documents
#topics, probs = topic_model.fit_transform(docs)


In [26]:
#Export and print details for each topic discovered by BERTopic
#The details will be Topic number, Counts (number of documents assigned to that topic)
#Name (abbreviated description providing top words for every topic)
#Representation (top words for that topic)
#Representative_Docs (a list of 2-3 document examples for that topic)

freq_df = topic_model.get_topic_info()
freq_df.to_excel('Topics.xlsx', sheet_name='Sheet1', index=False)
print("DataFrame successfully exported")

In [27]:
#Export Assigned Topics for each Article Abstract Sentence
#Adds two more columns to the original dataset
#BERTopic:  the topic number to which the document was assigned
#BERTopic Probability: the probability that the assignment is correct
#Note:  no entries will be added for the Assigned Sections that were filered out earlier in this notebook

data_filtered['Assigned Topic'] = topics
data_filtered['Assignment Probability'] = probs
data.loc[data_filtered.index, "BERTopic"] = data_filtered['Assigned Topic']
data.loc[data_filtered.index, "BERTopic Probability"] = data_filtered['Assignment Probability']

data.to_excel('TopicAssignments.xlsx', sheet_name='Sheet1', index=False)
print("DataFrame successfully exported")

In [28]:
#Explore BERTopic Results via Intertopic Distance Map
#Hover over clusters to discover which topics are assigned where in the map
topic_model.visualize_topics()

In [29]:
#Explore BERTopic Results using a Hierarchical Clustering Map
#This map provides suggestions as to how the topics might be combined into larger categories
hierarchical_topics = topic_model.hierarchical_topics(docs)
topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)


# Part 1c: Outliers
Documents assigned to Topic -1 are outliers (HDBSCAN did not place them in any cluster).

In [30]:
#Outliers
#Flag documents that BERTopic did not assign to any topic (Topic -1)
outliers = data_filtered['Assigned Topic'] == -1

print(f"Outlying responses: {outliers.sum()}")
print(data_filtered.loc[outliers, 'Document'])

# Part 2: Seeded BERTopic Model
Option 4 from the BERTopic notebook, seeded with Chickering and Gamson's Seven Principles for Good Practice in Undergraduate Education. Each seed list contains words for one principle. Note: seed_topic_list must be passed to BERTopic for seeding to take effect.

In [31]:
##Option 4:
#Seed topics: one list per principle in Chickering and Gamson's Seven Principles
seed_topic_list = [["office", "hours", "available", "email", "respond", "contact"],    #1 Student-faculty contact
                   ["group", "groups", "peers", "discussion", "together", "collaborate"], #2 Cooperation among students
                   ["practice", "problems", "examples", "hands", "work", "lab"],        #3 Active learning
                   ["feedback", "grading", "grade", "comments", "graded"],             #4 Prompt feedback
                   ["time", "review", "exam", "exams", "prepare", "quiz"],             #5 Time on task
                   ["expectations", "expect", "standards", "challenge"],               #6 High expectations
                   ["explain", "different", "ways", "patient", "understand", "visuals"]] #7 Diverse talents and ways of learning
hdbscan_model = HDBSCAN(
    min_cluster_size=10,
    prediction_data=True,
    )
vectorizer_model = CountVectorizer(stop_words="english")
seeded_topic_model = BERTopic(seed_topic_list=seed_topic_list, hdbscan_model=hdbscan_model, vectorizer_model=vectorizer_model, embedding_model="all-MiniLM-L6-v2")
##Fit the model and transform the documents
seeded_topics, seeded_probs = seeded_topic_model.fit_transform(docs)

In [32]:
#Export and print details for each topic discovered by the Seeded BERTopic model
#The details will be Topic number, Counts (number of documents assigned to that topic)
#Name (abbreviated description providing top words for every topic)
#Representation (top words for that topic)
#Representative_Docs (a list of 2-3 document examples for that topic)

seeded_freq_df = seeded_topic_model.get_topic_info()
seeded_freq_df.to_excel('Topics_Seeded.xlsx', sheet_name='Sheet1', index=False)
print("DataFrame successfully exported")

In [33]:
#Export Assigned Topics from the Seeded BERTopic model for each document
#Adds two more columns to the original dataset
#Seeded BERTopic:  the topic number to which the document was assigned
#Seeded BERTopic Probability: the probability that the assignment is correct

data_filtered['Seeded Assigned Topic'] = seeded_topics
data_filtered['Seeded Assignment Probability'] = seeded_probs
data.loc[data_filtered.index, "Seeded BERTopic"] = data_filtered['Seeded Assigned Topic']
data.loc[data_filtered.index, "Seeded BERTopic Probability"] = data_filtered['Seeded Assignment Probability']

data.to_excel('TopicAssignments_Seeded.xlsx', sheet_name='Sheet1', index=False)
print("DataFrame successfully exported")

In [34]:
#Outliers
#Flag documents that the Seeded BERTopic model did not assign to any topic (Topic -1)
seeded_outliers = data_filtered['Seeded Assigned Topic'] == -1

print(f"Outlying responses: {seeded_outliers.sum()}")
print(data_filtered.loc[seeded_outliers, 'Document'])

In [35]:
#Explore Seeded BERTopic Results via Intertopic Distance Map
#Hover over clusters to discover which topics are assigned where in the map
seeded_topic_model.visualize_topics()

In [36]:
#Explore Seeded BERTopic Results using a Hierarchical Clustering Map
#This map provides suggestions as to how the topics might be combined into larger categories
seeded_hierarchical_topics = seeded_topic_model.hierarchical_topics(docs)
seeded_topic_model.visualize_hierarchy(hierarchical_topics=seeded_hierarchical_topics)